In [5]:
!pip install ucimlrepo -q

In [6]:
import numpy as np
import pandas as pd
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [7]:
print("Download Concrete Compressive Strength dataset")
concrete_dataset = fetch_ucirepo(id=165)
X = concrete_dataset.data.features
y = concrete_dataset.data.targets.iloc[:, 0]

Download Concrete Compressive Strength dataset


In [15]:
X.head()

,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360


In [16]:
y.head()

,Concrete compressive strength
0,79.99
1,61.89
2,40.27
3,41.05
4,44.30


In [18]:
print("X size:", X.shape)
print("y size:", y.shape)

print("\n--- information ---")
X.info()

print("\n--- Missing values ---")
print(X.isna().sum())

print("\nMissing values in target:")
print(y.isna().sum())

print("\n--- check duplicate ---")
print(X.duplicated().sum())

display(X.describe().T)
display(y.describe().to_frame().T)

X size: (1030, 8)
y size: (1030,)

--- information ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1030 entries, 0 to 1029
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Cement              1030 non-null   float64
 1   Blast Furnace Slag  1030 non-null   float64
 2   Fly Ash             1030 non-null   float64
 3   Water               1030 non-null   float64
 4   Superplasticizer    1030 non-null   float64
 5   Coarse Aggregate    1030 non-null   float64
 6   Fine Aggregate      1030 non-null   float64
 7   Age                 1030 non-null   int64  
dtypes: float64(7), int64(1)
memory usage: 64.5 KB

--- Missing values ---
Cement                0
Blast Furnace Slag    0
Fly Ash               0
Water                 0
Superplasticizer      0
Coarse Aggregate      0
Fine Aggregate        0
Age                   0
dtype: int64

Missing values in target:
0

--- check duplicate ---
38


,count,mean,std,min,25%,50%,75%,max
Cement,1030.0,281.167864,104.506364,102.0,192.375,272.9,350.00,540.0
Blast Furnace Slag,1030.0,73.895825,86.279342,0.0,0.000,22.0,142.95,359.4
Fly Ash,1030.0,54.188350,63.997004,0.0,0.000,0.0,118.30,200.1
Water,1030.0,181.567282,21.354219,121.8,164.900,185.0,192.00,247.0
Superplasticizer,1030.0,6.204660,5.973841,0.0,0.000,6.4,10.20,32.2
Coarse Aggregate,1030.0,972.918932,77.753954,801.0,932.000,968.0,1029.40,1145.0
Fine Aggregate,1030.0,773.580485,80.175980,594.0,730.950,779.5,824.00,992.6
Age,1030.0,45.662136,63.169912,1.0,7.000,28.0,56.00,365.0


,count,mean,std,min,25%,50%,75%,max
Concrete compressive strength,1030.0,35.817961,16.705742,2.33,23.71,34.445,46.135,82.6


In [21]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train (use CV): {X_train.shape[0]} samples")
print(f"Test  (use to check): {X_test.shape[0]} samples\n")

Train (use CV): 824 samples
Test  (use to check): 206 samples



In [22]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Linear Regression (Baseline)': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
    'SVR': SVR(C=10.0, epsilon=0.1)
}

scoring = {
    'mae': 'neg_mean_absolute_error',
    'rmse': 'neg_root_mean_squared_error',
    'r2': 'r2'
}

In [25]:
cv_results = []

for name, model in models.items():
    for use_pca in [False, True]:
        steps = [('scaler', StandardScaler())]
        if use_pca:
            steps.append(('pca', PCA(n_components=0.95, random_state=42)))

        steps.append(('regressor', model))
        pipe = Pipeline(steps)
        scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring)

        mae_scores = -scores['test_mae']
        rmse_scores = -scores['test_rmse']
        r2_scores = scores['test_r2']

        cv_results.append({
            'Model': name,
            'PCA': 'Có (95%)' if use_pca else 'Không',
            'MAE Mean': mae_scores.mean(),
            'MAE Std': mae_scores.std(),
            'RMSE Mean': rmse_scores.mean(),
            'RMSE Std': rmse_scores.std(),
            'R² Mean': r2_scores.mean(),
            'R² Std': r2_scores.std()
        })


df_cv = pd.DataFrame(cv_results)
df_cv = df_cv.sort_values(by='R² Mean', ascending=False).reset_index(drop=True)

print(df_cv.to_string(index=False))

                       Model      PCA  MAE Mean  MAE Std  RMSE Mean  RMSE Std  R² Mean   R² Std
                     XGBoost    Không  3.335456 0.251391   5.105051  0.411345 0.906569 0.016741
               Random Forest    Không  3.721737 0.236555   5.337538  0.389884 0.898189 0.016001
                         SVR    Không  5.218531 0.441389   7.244116  0.559465 0.812522 0.028963
                         SVR Có (95%)  5.804503 0.417135   7.876749  0.570430 0.778526 0.031693
                     XGBoost Có (95%)  5.581032 0.429040   8.040685  0.860524 0.768545 0.044325
               Random Forest Có (95%)  5.987301 0.422761   8.216123  0.843594 0.759242 0.041481
Linear Regression (Baseline)    Không  8.421618 0.406350  10.627062  0.477243 0.597135 0.047330
Linear Regression (Baseline) Có (95%)  9.017666 0.407222  11.269571  0.520689 0.546729 0.054936


In [31]:
# test best model for test dataset
best_model_name = 'XGBoost'
best_steps = [('scaler', StandardScaler())]
best_steps.append(('regressor', models[best_model_name]))

best_pipeline = Pipeline(best_steps)
best_pipeline.fit(X_train, y_train)
y_test_pred = best_pipeline.predict(X_test)

test_mae = mean_absolute_error(y_test, y_test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_r2 = r2_score(y_test, y_test_pred)

df_test = pd.DataFrame([{
    'Model': best_model_name,
    'PCA': "Không",
    'Test MAE': test_mae,
    'Test RMSE': test_rmse,
    'Test R²': test_r2
}])
print(df_test.to_string(index=False))

  Model   PCA  Test MAE  Test RMSE  Test R²
XGBoost Không  3.034498   4.620778 0.917138
